# IPL Cleaning and EDA

Notebook for cleaning the IPL ball-by-ball dataset and running exploratory data analysis.

## Import Required Libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

## Load Raw IPL Dataset

In [5]:
INPUT_PATH = "att_0_1778303821_c3a907.csv"
OUTPUT_PATH = "ipl_cleaned.csv"

df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(df.columns.tolist())
display(df.head())

Loaded: 289,673 rows x 30 columns
['match_id', 'date', 'season', 'event', 'venue', 'city', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'win_by_runs', 'win_by_wickets', 'player_of_match', 'innings', 'batting_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'runs_batter', 'runs_extras', 'runs_total', 'extras_wides', 'extras_noballs', 'extras_byes', 'extras_legbyes', 'wicket_kind', 'wicket_player_out']


,match_id,date,season,event,venue,city,team1,team2,toss_winner,toss_decision,...,non_striker,runs_batter,runs_extras,runs_total,extras_wides,extras_noballs,extras_byes,extras_legbyes,wicket_kind,wicket_player_out
0,1082591,2017-04-05,2017,Indian Premier League,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,...,S Dhawan,0,0,0,0,0,0,0,NaN,NaN
1,1082591,2017-04-05,2017,Indian Premier League,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,...,S Dhawan,0,0,0,0,0,0,0,NaN,NaN
2,1082591,2017-04-05,2017,Indian Premier League,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,...,S Dhawan,4,0,4,0,0,0,0,NaN,NaN
3,1082591,2017-04-05,2017,Indian Premier League,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,...,S Dhawan,0,0,0,0,0,0,0,NaN,NaN
4,1082591,2017-04-05,2017,Indian Premier League,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,...,S Dhawan,0,2,2,2,0,0,0,NaN,NaN


## Normalize Season Column

In [6]:
df["season"] = df["season"].astype(str).str.extract(r"(\d{4})").astype(int)

## Convert Date to Datetime

In [7]:
df["date"] = pd.to_datetime(df["date"])

## Standardize Team Names

In [8]:
TEAM_MAP = {
    "Rising Pune Supergiant": "Rising Pune Supergiants",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Kings XI Punjab": "Punjab Kings",
    "Delhi Daredevils": "Delhi Capitals",
}

TEAM_COLUMNS = ["team1", "team2", "toss_winner", "winner", "batting_team"]
for col in TEAM_COLUMNS:
    df[col] = df[col].replace(TEAM_MAP)

## Fill City Nulls from Venue

In [9]:
VENUE_CITY_MAP = {
    "Dubai International Cricket Stadium": "Dubai",
    "Sharjah Cricket Stadium": "Sharjah",
}

df["city"] = df.apply(
    lambda row: VENUE_CITY_MAP.get(row["venue"], row["city"]) if pd.isna(row["city"]) else row["city"],
    axis=1,
)

## Clean Win Columns

In [10]:
df["win_by_runs"] = df["win_by_runs"].fillna(0).astype(int)
df["win_by_wickets"] = df["win_by_wickets"].fillna(0).astype(int)

## Fill Player of Match Nulls

In [11]:
df["player_of_match"] = df["player_of_match"].fillna("Unknown")

## Fill Wicket Columns

In [12]:
df["wicket_kind"] = df["wicket_kind"].fillna("none")
df["wicket_player_out"] = df["wicket_player_out"].fillna("none")

## Create Win Type Column

In [13]:
def get_win_type(row):
    if row["winner"] == "no result":
        return "no result"
    if row["winner"] == "tie":
        return "tie"
    if row["win_by_runs"] > 0:
        return "runs"
    if row["win_by_wickets"] > 0:
        return "wickets"
    return "unknown"

df["win_type"] = df.apply(get_win_type, axis=1)

## Create Bowler Wicket Column

In [14]:
NON_BOWLER_WICKETS = {
    "run out",
    "retired hurt",
    "obstructing the field",
    "retired out",
}

df["is_bowler_wicket"] = df["wicket_kind"].apply(lambda x: x not in NON_BOWLER_WICKETS and x != "none")

## Create Super Over Column

In [15]:
df["is_super_over"] = df["innings"] > 2

## Create Phase Column

In [16]:
df["phase"] = pd.cut(df["over"].astype(int), bins=[-1, 5, 14, 19], labels=["Powerplay", "Middle", "Death"])

## Verify Data Quality

In [17]:
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]

if remaining_nulls.empty:
    print("Verification: PASS")
else:
    print("Verification: WARNING")
    display(remaining_nulls)

print(df.describe(include="all"))

Verification: PASS
            match_id                        date         season  \
count   2.896730e+05                      289673  289673.000000   
unique           NaN                         NaN            NaN   
top              NaN                         NaN            NaN   
freq             NaN                         NaN            NaN   
mean    9.654832e+05  2017-06-06 03:50:40.201883    2016.981341   
min     3.359820e+05         2008-04-18 00:00:00    2007.000000   
25%     5.483650e+05         2012-05-13 00:00:00    2012.000000   
50%     1.082626e+06         2017-04-30 00:00:00    2017.000000   
75%     1.304085e+06         2022-04-26 00:00:00    2022.000000   
max     1.529292e+06         2026-05-06 00:00:00    2026.000000   
std     3.911715e+05                         NaN       5.601741   

                        event         venue    city  \
count                  289673        289673  289673   
unique                      1            59      37   
top     Ind

## Save Cleaned Dataset

In [18]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Saved: {OUTPUT_PATH}")

Cleaned: 289,673 rows x 34 columns
Saved: ipl_cleaned.csv
